# Bulk RNA-seq Analysis Pipeline (General Purpose)

## Instructions

This notebook integrates a complete differential analysis workflow, supporting **2-group/multi-group** comparisons and **human/mouse** data switching.

### Quick Start
1. Modify variables in the **"Parameter Configuration"** section below
2. Run all cells in order
3. Results are automatically saved to `./1-DEG/`, `./2-GSEA/`, `./3-Visualization/`

### Analysis Pipeline
| Step | Content | Output |
|------|---------|--------|
| 1 | Data Loading & QC | PCA, sample distance heatmap |
| 2 | DESeq2 Differential Expression | DEG tables (CSV) |
| 3 | DEG Visualization | Volcano plots, heatmaps, bar plots |
| 4 | Pathway Enrichment | ORA (GO/KEGG), GSEA |
| 5 | GSVA Gene Set Enrichment | Heatmaps, box plots |
| 6 | Single-Gene Visualization | Bar plots + statistical tests |


## 1. Parameter Configuration

**Only modify this section; the rest auto-adapts.**

In [ ]:
# Clear environment before setting parameters
rm(list = ls())
options(stringsAsFactors = FALSE)

# ===================== Parameter Configuration =====================
# ---- 1.1 Species ----
SPECIES       <- "mouse"

# ---- 1.2 Data Input ----
INPUT_FILE    <- "./0-Data/featureCounts_merged_count.annot.tsv"
INPUT_FORMAT  <- "tsv"           # "tsv" or "excel"
GENE_NAME_COL <- "gene_name"     # Gene name column
BIOTYPE_COL   <- "gene_biotype"  # Gene biotype column (set NULL to skip)
BIOTYPE_FILTER<- "protein_coding"# Biotype to keep
COUNT_COLS    <- NULL            # NULL = auto-detect numeric count columns; or e.g. c(2:10)

# ---- 1.3 Experimental Design ----
SAMPLE_NAMES  <- c("EM_LPS_1", "EM_LPS_2", "EM_LPS_3",
                    "LPS_1", "LPS_2", "LPS_3",
                    "blank_1", "blank_2", "blank_3")
GROUPS        <- c(rep("EM_LPS", 3), rep("LPS", 3), rep("blank", 3))
GROUP_LEVELS  <- c("blank", "LPS", "EM_LPS")

# ---- 1.4 Sample QC and optional exclusion ----
# QC metrics are always reported. Samples are only removed when listed here.
SAMPLE_EXCLUDE <- character(0)   # e.g. c("sample_bad_1", "sample_bad_2")
MIN_LIBRARY_SIZE <- NULL         # e.g. 1e6; NULL = do not flag by this metric
MIN_DETECTED_GENES <- NULL       # e.g. 8000; NULL = do not flag by this metric
MAX_ZERO_FRACTION <- NULL        # e.g. 0.80; NULL = do not flag by this metric
MIN_MEDIAN_CORRELATION_Z <- -3   # Flag sample if median sample-correlation z-score is too low

# ---- 1.5 Comparisons ----
# Comparisons: each c("output_name", "treatment", "control")
COMPARISONS   <- list(
  c("LPS_vs_Blank",  "LPS",    "blank"),
  c("EM_vs_Blank",   "EM_LPS", "blank"),
  c("EM_vs_LPS",     "EM_LPS", "LPS")
)

# ---- 1.6 Thresholds ----
PADJ_THRESH   <- 0.05          # Backward-compatible default; synced from DEFAULT_THRESHOLD below
LOG2FC_THRESH <- 1             # Backward-compatible default; synced from DEFAULT_THRESHOLD below

THRESHOLD_GRID <- data.frame(
  name   = c("strict", "standard", "loose"),
  padj   = c(0.01, 0.05, 0.10),
  log2fc = c(1.5, 1.0, 0.5),
  stringsAsFactors = FALSE
)
DEFAULT_THRESHOLD <- "standard" # Used by volcano, DEG heatmap, TF, and summary headline

MIN_COUNT     <- 10              # Keep genes with count >= MIN_COUNT in enough samples
DESIGN_FORMULA <- ~ condition    # Edit to e.g. ~ batch + condition if colData includes batch
PAIRWISE_TEST_METHOD <- "t.test" # Used only for GSVA/single-gene visualization

# ---- 1.7 GSVA: custom gene sets ----
custom_gene_sets <- list(
  M1_markers            = c("Cd86", "Cd80", "Tnf", "Il1b", "Il6", "Nos2", "Cxcl10", "Cxcl9", "Stat1", "Irf5", "Cd68"),
  M2_markers            = c("Mrc1", "Arg1", "Cd163", "Msr1", "Il10", "Tgfb1", "Stat6", "Irf4", "Pparg", "Chil3", "Retnla"),
  Inflammatory_response = c("Il1b", "Il6", "Tnf", "Il12b", "Cxcl1", "Cxcl2", "Ccl2", "Ptgs2", "Nfkb1", "Nfkbia"),
  Anti_inflammatory     = c("Il10", "Tgfb1", "Il4ra", "Stat6", "Socs1", "Socs3")
)

# ---- 1.8 Key genes for single-gene visualization ----
KEY_GENES <- c("Tnf", "Il1b", "Il6", "Cxcl10", "Nos2", "Ptgs2", "Mrc1", "Arg1", "Cd163", "Il10")

# ---- 1.9 Advanced analysis switches ----
RUN_TF_ANALYSIS      <- FALSE  # Set TRUE to run TF activity analysis (requires dorothea + viper)
RUN_COMPARECLUSTER   <- TRUE   # Set FALSE to skip; auto-skips if < 3 groups
COMPARECLUSTER_ONTOLOGY <- "BP" # "BP", "MF", "CC", or "ALL"


cat("Configuration complete!\n")
cat("Species:", SPECIES, "| Samples:", length(SAMPLE_NAMES), "| Groups:", length(unique(GROUPS)), "\n")
if (!all(c("name", "padj", "log2fc") %in% colnames(THRESHOLD_GRID))) stop("THRESHOLD_GRID must contain name, padj, and log2fc columns.")
if (any(duplicated(THRESHOLD_GRID$name))) stop("THRESHOLD_GRID$name must be unique.")
if (!DEFAULT_THRESHOLD %in% THRESHOLD_GRID$name) stop("DEFAULT_THRESHOLD must be one of THRESHOLD_GRID$name.")
default_threshold_config <- THRESHOLD_GRID[THRESHOLD_GRID$name == DEFAULT_THRESHOLD, ]
PADJ_THRESH <- default_threshold_config$padj
LOG2FC_THRESH <- default_threshold_config$log2fc
cat("Comparisons:", length(COMPARISONS), "| Default threshold:", DEFAULT_THRESHOLD,
    "padj<", PADJ_THRESH, "& |log2FC|>", LOG2FC_THRESH, "\n")

## 2. Environment Setup & Library Loading

In [ ]:
# Install Bioconductor packages (uncomment on first run)
# if (!require("BiocManager", quietly = TRUE)) install.packages("BiocManager")
# org_pkg <- ifelse(SPECIES == "human", "org.Hs.eg.db", "org.Mm.eg.db")
# BiocManager::install(c("DESeq2", "clusterProfiler", org_pkg,
#                        "ComplexHeatmap", "EnhancedVolcano", "GSVA",
#                        "msigdbr", "enrichplot", "DOSE", "BiocParallel"))
# install.packages(c("ashr", "openxlsx", "tidyverse", "pheatmap", "ggpubr"))

# Load libraries
suppressPackageStartupMessages({
  library(DESeq2)
  library(clusterProfiler)
  if (SPECIES == "human") { library(org.Hs.eg.db); org_db <- org.Hs.eg.db } else { library(org.Mm.eg.db); org_db <- org.Mm.eg.db }
  library(ComplexHeatmap)
  library(circlize)
  library(matrixStats)
  library(EnhancedVolcano)
  library(GSVA)
  library(msigdbr)
  library(ggplot2)
  library(dplyr)
  library(tidyr)
  library(RColorBrewer)
  library(ggrepel)
  library(DOSE)
  library(enrichplot)
  library(ggpubr)
  library(tidyverse)
  library(data.table)
  library(pheatmap)
  library(ashr)
  library(BiocParallel)
  if (RUN_TF_ANALYSIS) {
    suppressPackageStartupMessages({
      library(dorothea)
      library(viper)
      library(limma)
    })
  }
})

LIB_DIR <- if (dir.exists("RNAseq_lib")) "RNAseq_lib" else "../RNAseq_lib"
#LIB_DIR <- "/Users/luye/Library/Mobile Documents/com~apple~CloudDocs/Projects/RNAseq-Templates/RNAseq_lib"
source(file.path(LIB_DIR, "plot_utils.R"))
source(file.path(LIB_DIR, "io_utils.R"))
source(file.path(LIB_DIR, "deg_utils.R"))
source(file.path(LIB_DIR, "enrichment_utils.R"))

THRESHOLD_GRID <- validate_threshold_grid(THRESHOLD_GRID, DEFAULT_THRESHOLD)
default_threshold_config <- THRESHOLD_GRID[THRESHOLD_GRID$name == DEFAULT_THRESHOLD, ]
PADJ_THRESH <- default_threshold_config$padj
LOG2FC_THRESH <- default_threshold_config$log2fc

cat("Working directory:", getwd(), "
")
cat("RNAseq_lib:", LIB_DIR, "
")
cat("Packages and helper functions loaded successfully!
")

In [ ]:
dir.create("1-DEG", showWarnings = FALSE)
dir.create("2-GSEA", showWarnings = FALSE)
dir.create("3-Visualization", showWarnings = FALSE)

cat("Output directories created:\n")
cat("  - ./1-DEG/\n")
cat("  - ./2-GSEA/\n")
cat("  - ./3-Visualization/\n")

## 3. Publication-Grade Visualization Theme

In [ ]:
n_groups <- length(unique(GROUPS))
group_colors <- make_group_colors(GROUP_LEVELS)
colors_direction <- c("UP" = "#d6604d", "DOWN" = "#4393c3", "Not_Sig" = "#999999")
theme_set(theme_publication())
cat("Visualization theme loaded.
")

## 4. Data Loading & Preprocessing

In [ ]:
# Load data
rawcount <- read_count_table(INPUT_FILE, INPUT_FORMAT)

cat("Raw data dimensions:", nrow(rawcount), "rows x", ncol(rawcount), "columns
")
cat("Column names:", paste(colnames(rawcount), collapse = ", "), "
")
rawcount[1:6, 1:min(6, ncol(rawcount))]


In [ ]:
# Filter by biotype
if (!is.null(BIOTYPE_COL) && BIOTYPE_COL %in% colnames(rawcount)) {
  rawcount <- rawcount[rawcount[[BIOTYPE_COL]] == BIOTYPE_FILTER, ]
  cat("After biotype filtering:", nrow(rawcount), "rows
")
}

count_col_names <- detect_count_columns(rawcount, GENE_NAME_COL, COUNT_COLS)
validate_sample_design(SAMPLE_NAMES, GROUPS, GROUP_LEVELS, COMPARISONS, count_col_names)
cat("Count columns:", paste(count_col_names, collapse = ", "), "
")


In [ ]:
# Build count matrix
countData <- build_count_matrix(rawcount, GENE_NAME_COL, count_col_names, SAMPLE_NAMES)
cat("Final count matrix:", nrow(countData), "genes x", ncol(countData), "samples
")
preview_count_matrix(countData, GENE_NAME_COL)


In [ ]:
# Create colData, run sample-level QC, optionally exclude samples, then filter low-count genes
group <- factor(GROUPS, levels = GROUP_LEVELS)
colData <- make_col_data(countData, SAMPLE_NAMES, GROUPS, GROUP_LEVELS)

sample_qc <- calculate_sample_qc(countData, colData, group_col = "condition")
sample_qc <- flag_sample_qc(
  sample_qc,
  min_library_size = MIN_LIBRARY_SIZE,
  min_detected_genes = MIN_DETECTED_GENES,
  max_zero_fraction = MAX_ZERO_FRACTION,
  min_median_correlation_z = MIN_MEDIAN_CORRELATION_Z
)
write.csv(sample_qc, "./3-Visualization/Sample_QC_metrics.csv", row.names = FALSE)

p_sample_qc <- plot_sample_qc_pdf(sample_qc, "./3-Visualization/Sample_QC_metrics.pdf", group_colors)
plot_sample_correlation_pdf(sample_qc, "./3-Visualization/Sample_correlation_heatmap_raw_counts.pdf")
print(p_sample_qc)

if (any(sample_qc$qc_flag)) {
  cat("QC-flagged samples (review before excluding):
")
  print(sample_qc[sample_qc$qc_flag, c("sample", "qc_reason")])
} else {
  cat("No samples flagged by configured sample QC thresholds.
")
}

if (length(SAMPLE_EXCLUDE) > 0) {
  exclusion_res <- apply_sample_exclusion(countData, colData, SAMPLE_EXCLUDE)
  countData <- exclusion_res$count_data
  colData <- exclusion_res$col_data
  group <- colData$condition
  cat("Excluded samples:", paste(exclusion_res$excluded_samples, collapse = ", "), "
")
}

filter_res <- filter_low_count_genes(countData, group, MIN_COUNT)
countData <- filter_res$count_data
min_replicates <- filter_res$min_replicates

cat("After sample QC/exclusion and gene filtering:", nrow(countData), "genes x", ncol(countData), "samples
")
cat("Low-count filter: count >=", MIN_COUNT, "in at least", min_replicates, "samples
")
print(colData)


## 5. DESeq2 Object & Quality Control

In [ ]:
# Create DESeqDataSet
dds <- DESeqDataSetFromMatrix(
  countData = countData,
  colData   = colData,
  design    = DESIGN_FORMULA
)

# VST transformation
vsd <- vst(dds, blind = TRUE)
vsd_mat <- assay(vsd)

cat("VST matrix:", dim(vsd_mat)[1], "x", dim(vsd_mat)[2], "\n")


In [ ]:
# PCA with 95% confidence ellipses when each group has enough replicates
p_pca <- plot_pca_pdf(vsd, GROUP_LEVELS, group_colors, "./3-Visualization/PCA_plot.pdf")
print(p_pca)
cat("PCA plot saved.
")


In [ ]:
# Sample Distance Heatmap
p_sample_dist <- plot_sample_distance_pdf(vsd, "./3-Visualization/Sample_distance_heatmap.pdf")
cat("Sample distance heatmap saved.
")


In [ ]:
# Save intermediate data
save(rawcount, countData, group, colData, sample_qc, dds, vsd, vsd_mat, file = "./1-DEG/step0-input.Rdata")
cat("Intermediate data saved.\n")


## 6. DESeq2 Differential Expression Analysis

In [ ]:
# Run DESeq2
cat("Running DESeq2...\n")
dds <- DESeq(dds)
cat("DESeq2 analysis complete!\n")


In [ ]:
# Extract DESeq2 results and apply multiple DEG thresholds
res_list <- extract_deseq2_results(
  dds = dds,
  comparisons = COMPARISONS,
  condition_col = "condition",
  alpha = max(THRESHOLD_GRID$padj),
  shrink_type = "ashr"
)

deg_by_threshold <- build_deg_threshold_sets(res_list, THRESHOLD_GRID)
default_res_list <- get_threshold_result(deg_by_threshold, DEFAULT_THRESHOLD)
deg_summary <- write_deg_threshold_outputs(deg_by_threshold, outdir = "./1-DEG")

# Save combined results
save(res_list, deg_by_threshold, deg_summary, dds, vsd, vsd_mat, file = "./1-DEG/DEG_results.Rdata")
cat("
All DEG results saved to ./1-DEG/ with threshold-specific subdirectories.
")
print(deg_summary)


## 7. DEG Statistics Visualization

In [ ]:
# Summarize DEG counts across thresholds
print(deg_summary)
p_deg <- plot_deg_summary_pdf(deg_summary, "./3-Visualization/DEG_threshold_summary_barplot.pdf")
print(p_deg)
cat("DEG threshold summary plot saved.
")


## 8. Volcano Plot Visualization

In [ ]:
# Generate volcano plots for each comparison using DEFAULT_THRESHOLD
for (comp_name in names(res_list)) {
  p_vol <- plot_volcano_pdf(
    res_list[[comp_name]],
    comp_name = paste0(comp_name, " (", DEFAULT_THRESHOLD, ")"),
    padj_thresh = PADJ_THRESH,
    log2fc_thresh = LOG2FC_THRESH,
    filename = paste0("./3-Visualization/Volcano_", comp_name, "_", DEFAULT_THRESHOLD, ".pdf")
  )
  print(p_vol)
  cat("Volcano plot saved:", comp_name, "
")
}


## 9. Heatmap Visualization

In [ ]:
# ---- 9.1 Top variable genes heatmap ----
top_genes_mad <- head(order(rowMads(vsd_mat), decreasing = TRUE), 1000)

# Sample annotation
sample_anno <- data.frame(
  Group = factor(GROUPS, levels = GROUP_LEVELS)
)
rownames(sample_anno) <- colnames(vsd_mat)

col_fun <- colorRamp2(c(-2, 0, 2), c("#2166ac", "white", "#b2182b"))

ha_top <- HeatmapAnnotation(
  Group = sample_anno$Group,
  col = list(Group = group_colors)
)

pdf("./3-Visualization/Heatmap_top1000_variable_genes.pdf", width = 8, height = 10)
Heatmap(t(scale(t(vsd_mat[top_genes_mad, ]))),
        name = "Z-score",
        col = col_fun,
        top_annotation = ha_top,
        show_column_names = FALSE,
        show_row_names = FALSE,
        column_split = factor(GROUPS, levels = GROUP_LEVELS),
        cluster_column_slices = FALSE,
        use_raster = TRUE)
dev.off()
cat("Top 1000 variable genes heatmap saved.\n")


In [ ]:
# ---- 9.2 Top DEGs heatmap (from each comparison) ----
all_sig_genes <- unique(unlist(lapply(default_res_list, function(res) {
  res %>%
    filter(padj < PADJ_THRESH, abs(log2FoldChange) > LOG2FC_THRESH) %>%
    arrange(padj) %>%
    head(30) %>%
    pull(gene_name)
})))

sig_genes_in_mat <- intersect(all_sig_genes, rownames(vsd_mat))

cat("Top DEGs for heatmap:", length(sig_genes_in_mat), "genes\n")

if (length(sig_genes_in_mat) > 0) {
  pdf("./3-Visualization/Heatmap_topDEGs.pdf", width = 10, height = 12)
  Heatmap(t(scale(t(vsd_mat[sig_genes_in_mat, ]))),
          name = "Z-score",
          col = col_fun,
          top_annotation = ha_top,
          cluster_columns = FALSE,
          column_split = factor(GROUPS, levels = GROUP_LEVELS),
          cluster_column_slices = FALSE,
          row_names_gp = gpar(fontsize = 7),
          column_names_gp = gpar(fontsize = 9),
          column_title = "Top Differentially Expressed Genes")
  dev.off()
  cat("Top DEGs heatmap saved.\n")
}


## 10. Pathway Enrichment Analysis (clusterProfiler)

### 10.1 ORA - GO Enrichment

In [ ]:
# ---- Prepare gene lists for enrichment ----
# ORA uses THRESHOLD_GRID. GSEA uses the full ranked gene list and is not repeated by threshold.
ranked_lists <- lapply(res_list, ranked_gene_list)
background_entrez <- map_symbols_to_entrez(rownames(countData), org_db)
background_universe <- unique(background_entrez$ENTREZID)
org_code <- ifelse(SPECIES == "human", "hsa", "mmu")

cat("Mapped enrichment background:", length(background_universe), "Entrez IDs
")
cat("Prepared ranked gene lists for", length(ranked_lists), "comparisons.
")


In [ ]:
# ---- Multi-threshold GO/KEGG ORA for each comparison ----
ora_threshold <- run_threshold_ora(
  res_list = res_list,
  threshold_grid = THRESHOLD_GRID,
  org_db = org_db,
  universe = background_universe,
  organism = org_code,
  outdir = "./2-GSEA",
  plotdir = "./3-Visualization"
)
ora_summary <- ora_threshold$summary
write.csv(ora_summary, "./2-GSEA/ORA_threshold_summary.csv", row.names = FALSE)
cat("Multi-threshold ORA complete.
")
print(ora_summary)


### 10.2 ORA - KEGG Enrichment

In [ ]:
# ---- ORA summary visualization ----
if (exists("ora_summary") && nrow(ora_summary) > 0) {
  ora_long <- ora_summary %>%
    pivot_longer(cols = c("GO_terms", "KEGG_terms"), names_to = "Database", values_to = "Terms")
  p_ora_summary <- ggplot(ora_long, aes(x = Comparison, y = Terms, fill = Database)) +
    geom_col(position = "dodge", width = 0.7) +
    facet_wrap(~ Threshold, scales = "free_x") +
    labs(x = NULL, y = "Significant ORA terms", title = "ORA Term Counts Across DEG Thresholds") +
    theme_publication(base_size = 10) +
    theme(axis.text.x = element_text(angle = 35, hjust = 1), legend.position = "top")
  print(p_ora_summary)
  ggsave("./3-Visualization/ORA_threshold_summary_barplot.pdf", plot = p_ora_summary, width = 10, height = 5)
}


### 10.3 GSEA - GO & KEGG

In [ ]:
# ---- GSEA for each comparison (full ranked list; not threshold-dependent) ----
gsea_results <- list()
for (comp_name in names(ranked_lists)) {
  geneList <- ranked_lists[[comp_name]]
  entrezList <- make_entrez_ranked_list(geneList, org_db)

  if (length(entrezList) > 10) {
    ggo <- run_go_gsea(entrezList, org_db = org_db, ont = "ALL", min_size = 10, max_size = 500, p_cutoff = 0.05)
    if (!is.null(ggo) && nrow(as.data.frame(ggo)) > 0) {
      write.csv(as.data.frame(ggo), paste0("./2-GSEA/GSEA_GO_", comp_name, ".csv"), row.names = FALSE)
      plot_gsea_suite_pdf(ggo, paste0("./3-Visualization/GSEA_GO_", comp_name), paste("GSEA GO -", comp_name))
      cat("GSEA GO done:", comp_name, "-", nrow(as.data.frame(ggo)), "terms
")
    }

    gkegg <- run_kegg_gsea(entrezList, organism = org_code, min_size = 10, max_size = 500, p_cutoff = 0.05)
    if (!is.null(gkegg) && nrow(as.data.frame(gkegg)) > 0) {
      write.csv(as.data.frame(gkegg), paste0("./2-GSEA/GSEA_KEGG_", comp_name, ".csv"), row.names = FALSE)
      plot_gsea_suite_pdf(gkegg, paste0("./3-Visualization/GSEA_KEGG_", comp_name), paste("GSEA KEGG -", comp_name))
      cat("GSEA KEGG done:", comp_name, "-", nrow(as.data.frame(gkegg)), "terms
")
    }
    gsea_results[[comp_name]] <- list(go = ggo, kegg = gkegg)
  }
}
cat("
All GSEA analyses complete!
")


### 10.4 Multi-Group Comparison Enrichment (compareCluster)

In [ ]:
# ---- CompareCluster: compare enrichment across multiple comparisons ----
# This is most useful when you have > 2 groups.
# It produces a single dotplot showing how different comparisons enrich different pathways.

n_groups <- length(unique(GROUPS))
if (!RUN_COMPARECLUSTER || n_groups < 3) {
  cat("CompareCluster skipped:")
  if (!RUN_COMPARECLUSTER) cat(" RUN_COMPARECLUSTER is FALSE.\n")
  if (n_groups < 3) cat(" Only", n_groups, "group(s) &lt; 3. CompareCluster is designed for multi-group designs.\n")
} else {
  cat("Running CompareCluster for", length(res_list), "comparisons...\n")
}


In [ ]:
# ---- Prepare ENTREZID lists for compareCluster using DEFAULT_THRESHOLD ----
if (RUN_COMPARECLUSTER && n_groups >= 3) {
  cp_up <- list()
  cp_down <- list()

  for (comp_name in names(res_list)) {
    ed <- genes_for_enrichment(res_list[[comp_name]], PADJ_THRESH, LOG2FC_THRESH)

    if (length(ed$up) >= 5) {
      up_entrez <- map_symbols_to_entrez(ed$up, org_db)
      if (nrow(up_entrez) >= 5) cp_up[[comp_name]] <- up_entrez$ENTREZID
    }

    if (length(ed$down) >= 5) {
      down_entrez <- map_symbols_to_entrez(ed$down, org_db)
      if (nrow(down_entrez) >= 5) cp_down[[comp_name]] <- down_entrez$ENTREZID
    }
  }

  cat("UP gene lists for compareCluster:", length(cp_up), "comparisons
")
  cat("DOWN gene lists for compareCluster:", length(cp_down), "comparisons
")
}


In [ ]:
# ---- compareCluster GO ----
run_comparecluster_go <- function(gene_lists, direction_label) {
  if (length(gene_lists) < 2) return(NULL)
  mg_go <- compareCluster(
    gene_lists,
    fun = "enrichGO",
    OrgDb = org_db,
    universe = background_universe,
    ont = COMPARECLUSTER_ONTOLOGY,
    pAdjustMethod = "BH",
    pvalueCutoff = 0.1,
    readable = TRUE
  )
  if (!is.null(mg_go) && nrow(as.data.frame(mg_go)) > 0) {
    write.csv(as.data.frame(mg_go), paste0("./2-GSEA/CompareCluster_GO_", direction_label, ".csv"), row.names = FALSE)
    p_cc_go <- dotplot(mg_go, showCategory = 10, includeAll = TRUE,
                        title = paste("GO Enrichment -", direction_label, "genes")) +
      theme_publication(base_size = 10)
    ggsave(paste0("./3-Visualization/CompareCluster_GO_", direction_label, ".pdf"), plot = p_cc_go, width = 10, height = 12)
    print(p_cc_go)
    cat("CompareCluster GO (", direction_label, ") done:", nrow(as.data.frame(mg_go)), "terms\n")
  }
  mg_go
}

if (RUN_COMPARECLUSTER && n_groups >= 3) {
  mg_go_up <- run_comparecluster_go(cp_up, "up")
  mg_go_down <- run_comparecluster_go(cp_down, "down")
}


In [ ]:
# ---- compareCluster KEGG ----
run_comparecluster_kegg <- function(gene_lists, direction_label) {
  if (length(gene_lists) < 2) return(NULL)
  org_code <- ifelse(SPECIES == "human", "hsa", "mmu")
  mg_kegg <- compareCluster(
    gene_lists,
    fun = "enrichKEGG",
    organism = org_code,
    universe = background_universe,
    pAdjustMethod = "BH",
    pvalueCutoff = 0.1
  )
  if (!is.null(mg_kegg) && nrow(as.data.frame(mg_kegg)) > 0) {
    write.csv(as.data.frame(mg_kegg), paste0("./2-GSEA/CompareCluster_KEGG_", direction_label, ".csv"), row.names = FALSE)
    p_cc_kegg <- dotplot(mg_kegg, showCategory = 10, includeAll = TRUE,
                          title = paste("KEGG Enrichment -", direction_label, "genes")) +
      theme_publication(base_size = 10)
    ggsave(paste0("./3-Visualization/CompareCluster_KEGG_", direction_label, ".pdf"), plot = p_cc_kegg, width = 10, height = 10)
    print(p_cc_kegg)
    cat("CompareCluster KEGG (", direction_label, ") done:", nrow(as.data.frame(mg_kegg)), "terms\n")
  }
  mg_kegg
}

if (RUN_COMPARECLUSTER && n_groups >= 3) {
  mg_kegg_up <- run_comparecluster_kegg(cp_up, "up")
  mg_kegg_down <- run_comparecluster_kegg(cp_down, "down")
}


## 11. GSVA Gene Set Enrichment Analysis

In [ ]:
# Skip if no custom gene sets
gs_for_gsva <- list()
gsva_scores_df <- NULL
if (is.null(custom_gene_sets)) {
  cat("No custom gene sets provided. Skipping GSVA.\n")
} else {
  # Case-insensitive matching
  row_upper <- toupper(rownames(vsd_mat))
  upper_to_real <- setNames(rownames(vsd_mat), row_upper)

  gs_for_gsva <- list()
  for (gs_name in names(custom_gene_sets)) {
    genes <- custom_gene_sets[[gs_name]]
    matched <- intersect(toupper(genes), names(upper_to_real))
    found <- unique(upper_to_real[matched])
    if (length(found) >= 3) {
      gs_for_gsva[[gs_name]] <- found
      cat(gs_name, ":", length(found), "/", length(genes), "genes found\n")
    } else {
      cat(gs_name, ": skipped (only", length(found), "genes found)\n")
    }
  }

  if (length(gs_for_gsva) > 0) {
    params <- gsvaParam(
      as.matrix(vsd_mat), gs_for_gsva,
      minSize = 1, maxSize = Inf, kcdf = "Gaussian", tau = 1, maxDiff = TRUE
    )

    cat("Running GSVA...\n")
    gsva_scores <- gsva(params, verbose = FALSE, BPPARAM = SerialParam())
    gsva_scores_df <- as.data.frame(gsva_scores)

    write.csv(gsva_scores_df, "./2-GSEA/GSVA_scores.csv")
    cat("GSVA scores saved.\n")
  }
}


In [ ]:
# GSVA Heatmap
if (!is.null(custom_gene_sets) && length(gs_for_gsva) > 0) {
  col_fun_gsva <- colorRamp2(c(-0.5, 0, 0.5), c("#4393c3", "white", "#d6604d"))

  ha_gsva <- HeatmapAnnotation(
    Group = factor(GROUPS, levels = GROUP_LEVELS),
    col = list(Group = group_colors)
  )

  pdf("./3-Visualization/GSVA_heatmap.pdf", width = 8, height = max(3, length(gs_for_gsva) * 0.8))
  Heatmap(gsva_scores_df,
          name = "GSVA Score",
          col = col_fun_gsva,
          top_annotation = ha_gsva,
          cluster_rows = TRUE,
          show_row_dend = FALSE,
          cluster_columns = TRUE,
          show_column_dend = FALSE,
          show_column_names = FALSE,
          row_names_gp = gpar(fontsize = 10, fontface = "bold"),
          column_split = factor(GROUPS, levels = GROUP_LEVELS),
          cluster_column_slices = FALSE)
  dev.off()
  cat("GSVA heatmap saved.\n")
}


In [ ]:
# GSVA Boxplot
if (!is.null(custom_gene_sets) && length(gs_for_gsva) > 0) {
  sample_conditions <- data.frame(
    sample = colnames(vsd_mat),
    condition = factor(GROUPS, levels = GROUP_LEVELS)
  )

  gsva_long <- gsva_scores_df %>%
    as.data.frame() %>%
    mutate(Signature = rownames(gsva_scores_df)) %>%
    pivot_longer(cols = -Signature, names_to = "sample", values_to = "Score") %>%
    left_join(sample_conditions, by = "sample")

  p_gsva <- ggplot(gsva_long, aes(x = condition, y = Score, fill = condition)) +
    geom_boxplot(outlier.shape = NA, width = 0.6) +
    geom_jitter(width = 0.15, size = 2, color = "black", shape = 21, fill = "white") +
    facet_wrap(~ Signature, scales = "free_y") +
    stat_compare_means(method = PAIRWISE_TEST_METHOD, comparisons = combn(GROUP_LEVELS, 2, simplify = FALSE)) +
    labs(title = "GSVA Scores", x = NULL, y = "Score") +
    scale_fill_manual(values = group_colors) +
    theme_publication() +
    theme(legend.position = "none",
          strip.text = element_text(face = "bold", size = 11),
          plot.title = element_text(hjust = 0.5))

  print(p_gsva)
  ggsave("./3-Visualization/GSVA_boxplot.pdf", plot = p_gsva, width = 12, height = 3 * ceiling(length(gs_for_gsva) / 2))
  cat("GSVA boxplot saved.\n")
}


## 12. Single-Gene Expression Visualization

In [ ]:
# ---- SEM error bar functions ----
se_top <- function(x) mean(x) + sd(x) / sqrt(length(x))
se_bottom <- function(x) mean(x) - sd(x) / sqrt(length(x))

# ---- Plot function ----
plot_gene_expression <- function(gene_name, vsd_mat_obj, dds_obj) {
  if (!gene_name %in% rownames(vsd_mat_obj)) {
    cat("Gene", gene_name, "not found in expression matrix\n")
    return(NULL)
  }

  plot_data <- data.frame(
    sample = colnames(vsd_mat_obj),
    expression = as.numeric(vsd_mat_obj[gene_name, ]),
    condition = as.character(colData(dds_obj)[colnames(vsd_mat_obj), "condition"])
  )
  plot_data$condition <- factor(plot_data$condition, levels = GROUP_LEVELS)

  # Statistical comparisons
  comparisons <- combn(GROUP_LEVELS, 2, simplify = FALSE)
  stat_tbl <- compare_means(expression ~ condition, data = plot_data,
                            method = PAIRWISE_TEST_METHOD, comparisons = comparisons)

  ymax <- max(plot_data$expression, na.rm = TRUE)
  stat_tbl$y.position <- ymax * (1.08 + (seq_len(nrow(stat_tbl)) - 1) * 0.12)

  p <- ggplot(plot_data, aes(x = condition, y = expression, fill = condition)) +
    stat_summary(geom = "bar", fun = "mean", width = 0.60, color = "white", alpha = 0.95) +
    stat_summary(geom = "errorbar", fun.min = se_bottom, fun.max = se_top,
                 width = 0.20, linewidth = 0.45, color = "black") +
    geom_jitter(width = 0.12, size = 2.1, color = "black", alpha = 0.85, shape = 21, fill = "white") +
    stat_pvalue_manual(stat_tbl, label = "p.format", y.position = "y.position",
                       hide.ns = TRUE, tip.length = 0.01, bracket.size = 0.38) +
    labs(title = gene_name, x = NULL, y = "VST Expression") +
    scale_y_continuous(limits = c(0, NA), expand = expansion(mult = c(0, 0.13))) +
    theme_test(base_size = 13) +
    theme(legend.position = "none",
          axis.text = element_text(color = "black"),
          plot.title = element_text(hjust = 0.5, size = 15, face = "bold")) +
    scale_fill_manual(values = group_colors)

  p
}

# ---- Plot key genes ----
found_genes <- intersect(KEY_GENES, rownames(vsd_mat))
cat("Key genes found:", length(found_genes), "/", length(KEY_GENES), "\n")

for (gene in found_genes) {
  p_gene <- plot_gene_expression(gene, vsd_mat, dds)
  if (!is.null(p_gene)) {
    print(p_gene)
    ggsave(paste0("./3-Visualization/SingleGene_", gene, ".pdf"),
           plot = p_gene, width = 5, height = 6)
  }
}
cat("Single-gene plots saved.\n")


## 13. Results Summary

In [ ]:
# ---- Generate summary report ----
summary_report <- paste0(
  "========================================\n",
  "RNA-seq Analysis Summary Report\n",
  "========================================\n\n",
  "1. Data Overview\n",
  "   - Species: ", SPECIES, "\n",
  "   - Samples after optional QC exclusion: ", ncol(countData), "\n",
  "   - Samples manually excluded: ", ifelse(length(SAMPLE_EXCLUDE) == 0, "None", paste(SAMPLE_EXCLUDE, collapse = ", ")), "\n",
  "   - Groups: ", paste(unique(GROUPS), collapse = ", "), "\n",
  "   - DESeq2 design: ", deparse(DESIGN_FORMULA), "\n",
  "   - Low-count filter: count >= ", MIN_COUNT, " in at least ", min_replicates, " samples\n",
  "   - Genes after filtering: ", nrow(countData), "\n\n",
  "2. Differential Expression Default Threshold: ", DEFAULT_THRESHOLD,
  " (padj<", PADJ_THRESH, " & |log2FC|>", LOG2FC_THRESH, ")\n"
)

for (comp_name in names(default_res_list)) {
  res <- default_res_list[[comp_name]]
  n_up   <- sum(res$significance == "Up", na.rm = TRUE)
  n_down <- sum(res$significance == "Down", na.rm = TRUE)
  summary_report <- paste0(summary_report,
    "   - ", comp_name, ": ", n_up + n_down, " DEGs (Up: ", n_up, ", Down: ", n_down, ")\n")
}

summary_report <- paste0(summary_report, "\n3. Output Files\n")
summary_report <- paste0(summary_report, "   - ./1-DEG/<threshold>/DEG_results_*.csv\n")
summary_report <- paste0(summary_report, "   - ./1-DEG/DEG_threshold_summary.csv\n")
summary_report <- paste0(summary_report, "   - ./2-GSEA/<threshold>/GO_ORA_*.csv\n")
summary_report <- paste0(summary_report, "   - ./2-GSEA/<threshold>/KEGG_ORA_*.csv\n")
summary_report <- paste0(summary_report, "   - ./2-GSEA/ORA_threshold_summary.csv\n")
summary_report <- paste0(summary_report, "   - ./2-GSEA/GSEA_GO_*.csv\n")
summary_report <- paste0(summary_report, "   - ./2-GSEA/GSEA_KEGG_*.csv\n")
summary_report <- paste0(summary_report, "   - ./2-GSEA/GSVA_scores.csv\n")
summary_report <- paste0(summary_report, "   - ./3-Visualization/Sample_QC_metrics.csv\n")
summary_report <- paste0(summary_report, "   - ./3-Visualization/*.pdf (all plots)\n")
summary_report <- paste0(summary_report,
  "\n========================================\n")

cat(summary_report)
writeLines(summary_report, "./Analysis_summary.txt")
writeLines(capture.output(sessionInfo()), "./sessionInfo.txt")
cat("\nSummary report saved to ./Analysis_summary.txt\n")


## 14. Transcription Factor Activity Analysis (Optional)

Infer upstream transcription factor activity using `dorothea` regulons + `viper` algorithm, then identify differentially active TFs with `limma`.

In [ ]:
# ---- TF Analysis: check prerequisites ----
if (!RUN_TF_ANALYSIS) {
  cat("TF analysis skipped (RUN_TF_ANALYSIS = FALSE).\n")
} else {
  cat("Running transcription factor activity analysis...\n")
  cat("Species:", SPECIES, "| Comparisons:", length(COMPARISONS), "\n")
}


In [ ]:
# ---- Load species-specific dorothea regulon ----
if (RUN_TF_ANALYSIS) {
  if (SPECIES == "human") {
    data(dorothea_hs, package = "dorothea")
    regulon_df <- dorothea_hs
  } else {
    data(dorothea_mm, package = "dorothea")
    regulon_df <- dorothea_mm
  }

  # Keep high-confidence interactions (A, B, C)
  regulon_df <- regulon_df %>% filter(confidence %in% c("A", "B", "C"))
  regulon_list <- dorothea::df2regulon(regulon_df)

  cat("Regulon loaded:", length(regulon_list), "TFs,",
      nrow(regulon_df), "interactions (confidence A/B/C)\n")
}


In [ ]:
# ---- Infer TF activity with viper ----
if (RUN_TF_ANALYSIS) {
  tf_activity_matrix <- viper(
    eset = vsd_mat,
    regulon = regulon_list,
    nes = TRUE,
    method = "none",
    verbose = FALSE
  )

  cat("TF activity matrix:", nrow(tf_activity_matrix), "TFs x", ncol(tf_activity_matrix), "samples\n")
  write.csv(tf_activity_matrix, "./2-GSEA/TF_activity_matrix.csv")
}


In [ ]:
# ---- Differential TF activity (limma) ----
if (RUN_TF_ANALYSIS) {
  # Design matrix
  tf_design <- model.matrix(~0 + vsd$condition)
  colnames(tf_design) <- levels(vsd$condition)

  tf_res_list <- list()

  for (comp in COMPARISONS) {
    comp_name <- comp[1]
    treat     <- comp[2]
    ctrl      <- comp[3]

    # Check both groups exist in design
    if (!(treat %in% colnames(tf_design)) || !(ctrl %in% colnames(tf_design))) {
      cat("Skipping TF comparison", comp_name, ": group not found in design matrix\n")
      next
    }

    contrast_expr <- paste0(treat, " - ", ctrl)
    contrast_mat <- makeContrasts(contrasts = contrast_expr, levels = tf_design)

    fit <- lmFit(tf_activity_matrix, tf_design)
    fit2 <- contrasts.fit(fit, contrast_mat)
    fit2 <- eBayes(fit2)

    diff_tfs <- topTable(fit2, number = Inf, sort.by = "P")
    diff_tfs$TF <- rownames(diff_tfs)
    diff_tfs$significance <- ifelse(
      diff_tfs$adj.P.Val < PADJ_THRESH & abs(diff_tfs$logFC) > LOG2FC_THRESH,
      ifelse(diff_tfs$logFC > 0, "Up", "Down"),
      "Not_Sig"
    )

    tf_res_list[[comp_name]] <- diff_tfs
    write.csv(diff_tfs, paste0("./2-GSEA/TF_diff_activity_", comp_name, ".csv"), row.names = FALSE)

    n_sig <- sum(diff_tfs$significance != "Not_Sig", na.rm = TRUE)
    cat(comp_name, ":", n_sig, "differentially active TFs\n")
  }
}


In [ ]:
# ---- TF activity heatmap (top significant across all comparisons) ----
if (RUN_TF_ANALYSIS && length(tf_res_list) > 0) {
  # Collect top significant TFs from all comparisons
  top_tfs <- unique(unlist(lapply(tf_res_list, function(df) {
    df %>%
      filter(adj.P.Val < PADJ_THRESH, abs(logFC) > LOG2FC_THRESH) %>%
      head(15) %>%
      pull(TF)
  })))

  if (length(top_tfs) > 0) {
    tf_heatmap_mat <- tf_activity_matrix[intersect(top_tfs, rownames(tf_activity_matrix)), ]

    col_fun_tf <- colorRamp2(c(-2, 0, 2), c("#2166ac", "white", "#b2182b"))
    ha_tf <- HeatmapAnnotation(
      Group = factor(GROUPS, levels = GROUP_LEVELS),
      col = list(Group = group_colors)
    )

    pdf("./3-Visualization/TF_activity_heatmap.pdf", width = 8, height = max(4, length(top_tfs) * 0.3))
    Heatmap(tf_heatmap_mat,
            name = "TF Activity (NES)",
            col = col_fun_tf,
            top_annotation = ha_tf,
            cluster_rows = TRUE,
            show_row_dend = FALSE,
            cluster_columns = TRUE,
            show_column_dend = FALSE,
            show_column_names = FALSE,
            row_names_gp = gpar(fontsize = 9, fontface = "bold"),
            column_split = factor(GROUPS, levels = GROUP_LEVELS),
            cluster_column_slices = FALSE)
    dev.off()
    cat("TF activity heatmap saved:", nrow(tf_heatmap_mat), "TFs\n")
  } else {
    cat("No significant TFs found for heatmap.\n")
  }
}


In [ ]:
# ---- Top TF activity barplot (first comparison) ----
if (RUN_TF_ANALYSIS && length(tf_res_list) > 0) {
  comp_name <- names(tf_res_list)[1]
  diff_tfs <- tf_res_list[[comp_name]]

  # Top 10 up and top 10 down
  top_up   <- diff_tfs %>% filter(adj.P.Val < PADJ_THRESH, logFC > LOG2FC_THRESH) %>% head(10)
  top_down <- diff_tfs %>% filter(adj.P.Val < PADJ_THRESH, logFC < -LOG2FC_THRESH) %>% head(10)
  top_tf   <- rbind(top_up, top_down)

  if (nrow(top_tf) > 0) {
    top_tf$Regulation <- ifelse(top_tf$logFC > 0, "Up", "Down")

    p_tf_bar <- ggplot(top_tf, aes(x = reorder(TF, logFC), y = logFC, fill = Regulation)) +
      geom_bar(stat = "identity", width = 0.7) +
      coord_flip() +
      scale_fill_manual(values = colors_direction) +
      labs(title = paste("Top Differentially Active TFs -", comp_name),
           x = NULL, y = "Log2 Fold Change (TF Activity)") +
      theme_publication(base_size = 11) +
      theme(legend.position = "top")

    print(p_tf_bar)
    ggsave(paste0("./3-Visualization/TF_barplot_", comp_name, ".pdf"),
           plot = p_tf_bar, width = 7, height = 8)
    cat("TF barplot saved for", comp_name, "\n")
  }
}
